In [27]:
# 1. load an inspect the data
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


df = pd.read_csv('telco.csv')
print("shape:", df.shape)
print(df.head())
df.info()

# confirm target is MonthlyCharges and is continuous
# continuous if there are a lot of different unique numbers 
# and also if its a float or int 
print(df['target'].dtype)
print(df['target'].describe())
print(df['target'].nunique())  
# check for missing values for target 
print(df['target'].isnull().sum())  

# verify service columns are clean 0/1 indicators with no missing values 
service_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'target', 
                'TotalCharges', 'Churn', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 
                'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']  

for col in service_cols:
    # print out the distinct values using unique() and check for missing values using isnull()
    print(col)
    print(df[col].unique())
    print(df[col].isnull().sum()) 
    

shape: (7043, 24)
   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   
3       1              0        0           0      45             0   
4       0              0        0           0       2             1   

   MultipleLines  OnlineSecurity  OnlineBackup  DeviceProtection  ...  target  \
0              0               0             1                 0  ...   29.85   
1              0               1             0                 1  ...   56.95   
2              0               1             1                 0  ...   53.85   
3              0               1             0                 1  ...   42.30   
4              0               0             0                 0  ...   70.70   

   TotalCharges  Churn  InternetService_Fiber optic  InternetService

In [28]:
# 2. define the variables 

# set target as MonthlyCharges and predictors as ten service indicators (PhoneService, MultipleLines, OnlineSecurity, OnlineBackup, 
# DeviceProtection, TechSupport, StreamingTV, StreamingMovies, InternetService_Fiber optic, InternetService_No)

# rename target to be MonthlyCharges
df = df.rename(columns = {"target":"MonthlyCharges"})
target = 'MonthlyCharges' 
y = df[target]
print(y)

# define predictors
X = df[['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 
        'StreamingMovies', 'InternetService_Fiber optic', 'InternetService_No']]

print(X)

0        29.85
1        56.95
2        53.85
3        42.30
4        70.70
         ...  
7038     84.80
7039    103.20
7040     29.60
7041     74.40
7042    105.65
Name: MonthlyCharges, Length: 7043, dtype: float64
      PhoneService  MultipleLines  OnlineSecurity  OnlineBackup  \
0                0              0               0             1   
1                1              0               1             0   
2                1              0               1             1   
3                0              0               1             0   
4                1              0               0             0   
...            ...            ...             ...           ...   
7038             1              1               1             0   
7039             1              1               0             1   
7040             0              0               1             0   
7041             1              1               0             0   
7042             1              0              

In [29]:
# 3. split the data
#  hold out a test set (80/20) with a fixed random seed so results are reproducible

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [30]:
# 4. Fit a multivariable linear regression
# train the model on the training set 

# predicts MonthlyCharges using all 10 service columns individually
# as multiple variables. 
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [31]:
# 5. Report the parameters
print(f'Intercept: ${model.intercept_:.2f}')
print('Coefficients:')
# print out the array of coefficients, one for each column in the x training data
for name, coeficients in zip(X_train.columns, model.coef_):
    print(f'{name}: ${coeficients:.2f}')

Intercept: $24.97
Coefficients:
PhoneService: $20.04
MultipleLines: $5.01
OnlineSecurity: $5.05
OnlineBackup: $4.98
DeviceProtection: $5.01
TechSupport: $5.03
StreamingTV: $9.98
StreamingMovies: $9.95
InternetService_Fiber optic: $24.95
InternetService_No: $-25.05


In [32]:
# 6. Evaluate on the test set
y_prediction = model.predict(X_test)
# calculate the r2, mae and rmse scores 
# r^2 tells us what fraction of the variance in MonthlyCharges the 
# service features explain, so closer ot 1.0 is better and 0 means 
# that it is no better than jsut guessing the average
r2 = r2_score(y_test, y_prediction)
# mae means that the prediction is off by x amoutn of dollars on avg
mae = mean_absolute_error(y_test, y_prediction)
# rmse is in dollars and is the error squared before averaging so 
# larger misses are penalized more than smaller ones
# squared=False bc it returns the square root at the end 
rmse = mean_squared_error(y_test, y_prediction, squared=False)

print(f"R²:   {r2:.4f}")
print(f"MAE:  ${mae:.2f}")
print(f"RMSE: ${rmse:.2f}")

R²:   0.9988
MAE:  $0.79
RMSE: $1.05


In [33]:
# 7. Compare to a baseline

# define add ons and use same columns as X
addon_columns = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
              'InternetService_Fiber optic', 'InternetService_No']

# collapse the 10 columns into 1 
# predicts monthlyCharges from this one value instead of all 10 individual variables
df['addon_count'] = df[addon_columns].sum(axis=1)

# baseline predictor and keep same rows as X_train/X_test just now one column
X_train_baseline = df.loc[X_train.index, ['addon_count']]
X_test_baseline  = df.loc[X_test.index,  ['addon_count']]

# fit baseline model
baseline_model = LinearRegression()
baseline_model.fit(X_train_baseline, y_train)

y_prediction_baseline = baseline_model.predict(X_test_baseline)

# evaluate baseline
r2_base = r2_score(y_test, y_prediction_baseline)
mae_base = mean_absolute_error(y_test, y_prediction_baseline)
rmse_base = mean_squared_error(y_test, y_prediction_baseline, squared=False)

print("Baseline (addon_count only):")
print(f"  R²:   {r2_base:.4f}")
print(f"  MAE:  ${mae_base:.2f}")
print(f"  RMSE: ${rmse_base:.2f}")

print("\nMultivariable model:")
print(f"  R²:   {r2:.4f}")
print(f"  MAE:  ${mae:.2f}")
print(f"  RMSE: ${rmse:.2f}")

Baseline (addon_count only):
  R²:   0.7009
  MAE:  $14.24
  RMSE: $16.46

Multivariable model:
  R²:   0.9988
  MAE:  $0.79
  RMSE: $1.05


In [34]:
# 8. Interpret the results 
# convert to a tabel with the service and the monthly price of the service 
coeficient_table = pd.DataFrame({
    'service': X_train.columns,
    'monthly price': model.coef_
}).sort_values('monthly price', ascending=False)

print(f"Base charge (intercept): ${model.intercept_:.2f}")
print(coeficient_table.to_string(index=False))


Base charge (intercept): $24.97
                    service  monthly price
InternetService_Fiber optic      24.951892
               PhoneService      20.036039
                StreamingTV       9.980786
            StreamingMovies       9.946628
             OnlineSecurity       5.045880
                TechSupport       5.028676
              MultipleLines       5.012293
           DeviceProtection       5.011923
               OnlineBackup       4.979713
         InternetService_No     -25.047916


In [35]:
# 9. Make a prediction

# define a new customer's chosen bundle
# 1 = has the service, 0 = does not
new_customer = pd.DataFrame({
    'PhoneService': [1],
    'MultipleLines': [0],
    'OnlineSecurity': [1],
    'OnlineBackup': [0],
    'DeviceProtection': [0],
    'TechSupport': [1],
    'StreamingTV': [1],
    'StreamingMovies': [1],
    'InternetService_Fiber optic': [1],
    'InternetService_No': [0]
})

predicted_charge = model.predict(new_customer)
print(f"Expected monthly charge: ${predicted_charge[0]:.2f}")

Expected monthly charge: $99.96
